In [ ]:
!pip install amrlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q amrlib
!python -m spacy download en_core_web_sm

In [ ]:
!pip install unidecode penman torch torchvision torchaudio --quiet

In [ ]:
import amrlib
import tarfile
import urllib.request
from pathlib import Path

amrlib_dir = Path(amrlib.__file__).resolve().parent
data_dir = amrlib_dir / "data"
data_dir.mkdir(exist_ok=True)

# Correct GitHub release URL: uses the TAG name, not the release title
url = "https://github.com/bjascob/amrlib-models/releases/download/parse_xfm_bart_large-v0_1_0/model_parse_xfm_bart_large-v0_1_0.tar.gz"

tar_path = data_dir / "model_parse_xfm_bart_large-v0_1_0.tar.gz"
extracted_dir = data_dir / "model_parse_xfm_bart_large-v0_1_0"
target_dir = data_dir / "model_stog"

if not tar_path.exists():
    print("Downloading model...")
    urllib.request.urlretrieve(url, tar_path)

if not extracted_dir.exists():
    print("Extracting model...")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=data_dir)

if not target_dir.exists():
    extracted_dir.rename(target_dir)

print("Installed parser model at:", target_dir)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
INPUT_FILE = "/content/drive/My Drive/COMP459/outputs/post_processed_segmentation_all.jsonl"
!ls "{INPUT_FILE}"

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Iterable, List

SEG_TOKEN = "_seg_"
WHITESPACE_RE = re.compile(r"\s+")

OUTPUT_FILE = "/content/drive/My Drive/COMP459/outputs/all_data.jsonl"
BATCH_SIZE = 16

def batched(items: List[str], batch_size: int) -> Iterable[List[str]]:
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def parse_amrs(sentences: List[str], stog_model, batch_size: int = 16) -> List[str]:
    if not sentences:
        return []

    outputs: List[str] = []
    for batch in batched(sentences, batch_size):
        graphs = stog_model.parse_sents(batch)
        outputs.extend(graphs)
    return outputs

def run(
    input_file: str = INPUT_FILE,
    output_file: str = OUTPUT_FILE,
    batch_size: int = BATCH_SIZE,
) -> None:
    input_path = Path(input_file)
    output_path = Path(output_file)

    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    import amrlib

    print(f"Loading records from {input_path}...")
    records = []
    with input_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    if not records:
        print("No records found. Writing empty output.")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("w", encoding="utf-8") as f:
            pass
        return

    print("Loading AMR model...")
    stog_model = amrlib.load_stog_model()

    normal_texts: List[str] = []
    flat_easy_texts: List[str] = []
    easy_lens: List[int] = []

    for r in records:
        normal_texts.append(r.get("normal_text", ""))
        
        # Collect rewritten Easy Read sentences (or fallback to original segments)
        easy_sents = r.get("easy_read_rewritten")
        if not easy_sents:
            easy_sents = r.get("easy_text", [])
        flat_easy_texts.extend(easy_sents)
        easy_lens.append(len(easy_sents))

    print(f"Parsing {len(normal_texts)} normal sentence(s)...")
    normal_amrs = parse_amrs(normal_texts, stog_model, batch_size=batch_size)

    print(f"Parsing {len(flat_easy_texts)} easy sentence(s)...")
    flat_easy_amrs = parse_amrs(flat_easy_texts, stog_model, batch_size=batch_size)

    easy_idx = 0
    for i, r in enumerate(records):
        r["normal_amr"] = [normal_amrs[i]] if i < len(normal_amrs) and normal_amrs[i] else []
        n_easy = easy_lens[i]
        r["easy_amr"] = flat_easy_amrs[easy_idx : easy_idx + n_easy]
        easy_idx += n_easy

    print(f"Wrote {len(records)} record(s) to {output_path}...")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print("AMR Generation completed successfully.")

run()